In [1]:
import nfl_data_py as nfl
from pydantic import BaseModel, Field, ValidationError, conlist
from typing import List, Dict, Literal, Optional
import polars as pl
from typing import Tuple, Literal, Optional, Dict, Any
from __future__ import annotations
import re
import polars as pl
from metrics_executor import MetricsExecutor

In [2]:
#clean 2025 data

# Load 2025 play-by-play data
df_twofive = pl.from_pandas(
    nfl.import_pbp_data(
        [2025],
        downcast=True,
        cache=False,
        alt_path=None,
        include_participation=False
    )
)

# Keep only regular season plays
df_twofive = df_twofive.filter(pl.col("season_type") == "REG")

# Normalize pass/rush flags
df_twofive = df_twofive.with_columns([
    pl.col("pass").fill_null(0).cast(pl.Int8).alias("pass"),
    pl.col("rush").fill_null(0).cast(pl.Int8).alias("rush"),
])

# Overwrite play_type pass/run if flagged, else keep original
#     Also handle any numeric or null values 
df_twofive = df_twofive.with_columns(
    pl.when(pl.col("pass") == 1).then(pl.lit("pass"))
     .when(pl.col("rush") == 1).then(pl.lit("run"))
     .otherwise(
         pl.col("play_type")
           .cast(pl.Utf8)
           .fill_null("no_play")
           .replace({"0": "no_play", "0.0": "no_play"})
     )
     .alias("play_type")
)

#Clean up whitespace / case for consistency
df_twofive = df_twofive.with_columns(
    pl.col("play_type").str.strip_chars().str.to_lowercase()
)



df_twofive.head()

2025 done.
Downcasting floats.


play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,side_of_field,yardline_100,game_date,quarter_seconds_remaining,half_seconds_remaining,game_seconds_remaining,game_half,quarter_end,drive,sp,qtr,down,goal_to_go,time,yrdln,ydstogo,ydsnet,desc,play_type,yards_gained,shotgun,no_huddle,qb_dropback,qb_kneel,qb_spike,qb_scramble,pass_length,…,home_coach,away_coach,stadium_id,game_stadium,aborted_play,success,passer,passer_jersey_number,rusher,rusher_jersey_number,receiver,receiver_jersey_number,pass,rush,first_down,special,play,passer_id,rusher_id,receiver_id,name,jersey_number,id,fantasy_player_name,fantasy_player_id,fantasy,fantasy_id,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
f32,str,str,str,str,str,i32,str,str,str,str,f32,str,f32,f32,f32,str,f32,f32,f32,f32,f32,i32,str,str,f32,f32,str,str,f32,f32,f32,f32,f32,f32,f32,str,…,str,str,str,str,f32,f32,str,f32,str,f32,str,f32,i8,i8,f32,f32,f32,str,str,str,str,f32,str,str,str,str,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
1.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,null,null,null,null,null,"""2025-09-07""",900.0,1800.0,3600.0,"""Half1""",0.0,null,0.0,1.0,null,0,"""15:00""","""NO 35""",0.0,null,"""GAME""","""no_play""",null,0.0,0.0,null,0.0,0.0,0.0,null,…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,0.0,null,null,null,null,null,null,0,0,null,0.0,0.0,null,null,null,null,null,null,null,null,null,null,0.0,0.0,-0.0,null,null,null,null,null,null,null
40.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,"""ARI""","""away""","""NO""","""NO""",35.0,"""2025-09-07""",900.0,1800.0,3600.0,"""Half1""",0.0,1.0,0.0,1.0,null,0,"""15:00""","""NO 35""",0.0,2.0,"""19-B.Grupe kicks 65 yards from…","""kickoff""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,0.0,null,null,null,null,null,null,0,0,0.0,1.0,0.0,null,null,null,null,null,null,null,null,null,null,0.0,0.0,-0.3527,null,null,null,null,null,null,null
63.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,"""ARI""","""away""","""NO""","""ARI""",78.0,"""2025-09-07""",896.0,1796.0,3596.0,"""Half1""",0.0,1.0,0.0,1.0,1.0,0,"""14:56""","""ARI 22""",10.0,2.0,"""(14:56) 6-J.Conner right tackl…","""run""",3.0,0.0,0.0,0.0,0.0,0.0,0.0,null,…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,0.0,null,null,"""J.Conner""",6.0,null,null,0,1,0.0,0.0,1.0,null,"""00-0033553""",null,"""J.Conner""",6.0,"""00-0033553""","""J.Conner""","""00-0033553""","""J.Conner""","""00-0033553""",0.0,0.0,-0.190052,null,null,null,null,null,0.511128,-51.112808
85.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,"""ARI""","""away""","""NO""","""ARI""",75.0,"""2025-09-07""",858.0,1758.0,3558.0,"""Half1""",0.0,1.0,0.0,1.0,2.0,0,"""14:18""","""ARI 25""",7.0,2.0,"""(14:18) (Shotgun) 1-K.Murray p…","""pass""",11.0,1.0,0.0,1.0,0.0,0.0,0.0,"""short""",…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,1.0,"""K.Murray""",1.0,null,null,"""T.McBride""",85.0,1,0,1.0,0.0,1.0,"""00-0035228""",null,"""00-0037744""","""K.Murray""",1.0,"""00-0035228""","""T.McBride""","""00-0037744""","""T.McBride""","""00-0037744""",1.0,0.0,1.31734,0.939998,4.750889,3.0,0.666726,0.43911,0.66894,33.105968
115.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,"""ARI""","""away""","""NO""","""ARI""",64.0,"""2025-09-07""",820.0,1720.0,3520.0,"""Half1""",0.0,1.0,0.0,1.0,1.0,0,"""13:40""","""ARI 36""",10.0,2.0,"""(13:40) 1-K.Murray sacked at A…","""pass""",-11.0,0.0,0.0,1.0,0.0,0.0,0.0,null,…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,0.0,"""K.Murray""",1.0,null,null,null,null,1,0,0.0,0.0,1.0,"""00-0035228""",null,null,"""K.Murray""",1.0,"""00-0035228""",null,nul

In [3]:
ex = MetricsExecutor(df_twofive)

df_out, summary = ex.execute("ranked epa per play")  # offense, overall EPA/play
df_out, summary = ex.execute("ranked defensive epa per play allowed")
df_out, summary = ex.execute("ranked rushing epa per play")
df_out, summary = ex.execute("offensive success rate")
df_out, summary = ex.execute("defensive pass success rate")